# 02 - Train MiniConvNet (both architecture variants x both split variants)

**What this notebook does**: four training runs -

| run | architecture | split | class weights |
|---|---|---|---|
| A | `gap` (~106K params) | `faithful` | no |
| B | `flatten` (~0.5M params) | `faithful` | no |
| C | `gap` | `clean` | yes |
| D | `flatten` | `clean` | yes |

Run B is also followed by the model-file-size measurement for the paper's "~6MB" claim (LESSON 2).

**What must already exist**: the split CSVs from `00_dataset_audit.ipynb`. GPU accelerator should be
on (Kaggle: Settings -> Accelerator -> GPU T4 x2 / P100).

**Built-in safeguards**
* `compile_model()` applies `Adam(lr=1e-4, clipnorm=1.0)` **by default** (LESSON 1). Do not raise the
  learning rate to `1e-3` - that is exactly what produced dead-ReLU collapse before.
* `detect_collapse()` runs after **every** run (LESSON 3), and now also flags *partial* collapse
  (LESSON 11): a model that never predicts some classes at all is written out as
  `INVALID_partial_collapse` even when its kappa is nonzero.
* Raw per-sample predictions are saved to `outputs/predictions/` for every run (LESSON 11), so a
  diagnostic invented later never requires a retrain to apply.
* Both architecture variants are trained; neither is silently preferred (LESSON 2).
* `clean`-split results are a robustness experiment, **not** a replication of the paper (LESSON 4).

**Where results go**: all four runs are logged to `outputs/experiments_log.csv` with a `config_note`.
The canonical MiniConvNet number for the main table is the 5-fold CV mean +/- std from notebook 04
(LESSON 8) - single runs never go into `results_table.csv`.

**What "looks right"**: training and validation accuracy both rise above chance (0.25) within the
first ~10 epochs and the loss curve is not a flat line. A run whose loss is identical for 10+
consecutive epochs has collapsed. A run whose confusion matrix has an all-zero **column** has
partially collapsed - equally unreportable, and now caught automatically.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('models dir:', MODELS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import load_split, make_split_datasets, split_counts
from src.models import build_miniconvnet, count_params
from src.train_utils import (set_global_seeds, gpu_report, compile_model, optimizer_summary,
                            class_weights_for, make_callbacks, save_history, plot_history,
                            final_epoch_summary, run_name_for, checkpoint_path,
                            measure_model_file_sizes)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                               plot_confusion_matrix, per_class_report,
                               tumor_vs_subtype_breakdown, result_row_from_metrics,
                               record_experiment, save_predictions)

set_global_seeds(SEED)
print(gpu_report())
print('epochs:', EPOCHS_MINICONVNET, '| lr:', LR_MINICONVNET, '| clipnorm:', CLIPNORM)
print('backbone activation:', MINICONVNET_ACTIVATION,
      '(LESSON 11 fallback: set MINICONVNET_ACTIVATION = "leaky_relu" in src/config.py)')

## 1. Model shapes and parameter counts

Builds both variants once, purely to show what they are.

**Looks right**: `gap` ~106K parameters, `flatten` ~0.5M parameters. The gap between these two
numbers *is* the architecture ambiguity described in the README - the paper claims ~0.5M, which only
the `flatten` head reaches.

In [ ]:
for variant in ARCH_VARIANTS:
    m = build_miniconvnet(variant)
    print(f'--- {variant} ---')
    print(count_params(m))
    del m

In [ ]:
demo = build_miniconvnet('flatten')
demo.summary()
print('\nfeature map before the head:', demo.get_layer('pool4').output.shape,
      '-> Flatten width', int(np.prod(demo.get_layer('pool4').output.shape[1:])))
del demo

## 2. Run helpers

Three small functions, deliberately kept in the notebook so they can be edited while debugging:
`prepare_run` (data), `train_run` (fit), `evaluate_run` (metrics + collapse check + logging).

In [ ]:
def prepare_run(split_variant):
    """Load a split, build datasets, and report what was loaded."""
    sdf = load_split(split_variant)
    train_ds, val_ds, test_ds, frames = make_split_datasets(sdf)
    cw = class_weights_for(split_variant, frames['train']['label'].values)
    print(f'split_variant = {split_variant}')
    print(split_counts(sdf))
    print('class_weight:', cw if cw else 'None (not used for this split)')
    return {'df': sdf, 'train_ds': train_ds, 'val_ds': val_ds, 'test_ds': test_ds,
            'frames': frames, 'class_weight': cw}

In [ ]:
def train_run(arch_variant, split_variant, data, epochs=EPOCHS_MINICONVNET,
              dropout_rate=DROPOUT_RATE, run_suffix=None):
    """Build -> compile (collapse-safe defaults) -> fit -> save history."""
    set_global_seeds(SEED)
    run_name = run_name_for('miniconvnet', arch_variant, split_variant, run_suffix)

    model = build_miniconvnet(arch_variant, dropout_rate=dropout_rate)
    compile_model(model)                      # Adam(1e-4, clipnorm=1.0) by default
    print('run       :', run_name)
    print('params    :', count_params(model))
    print('optimizer :', optimizer_summary(model))

    history = model.fit(
        data['train_ds'], validation_data=data['val_ds'], epochs=epochs,
        class_weight=data['class_weight'],
        callbacks=make_callbacks(run_name), verbose=2)

    save_history(history, run_name)
    print('\n', final_epoch_summary(history))
    print('checkpoint:', checkpoint_path(run_name))
    return run_name, model, history

In [ ]:
def evaluate_run(run_name, model, history, data, arch_variant, split_variant,
                 config_note, kind='experiment'):
    """Test-set metrics + collapse check + confusion matrix + CSV logging."""
    y_true, y_pred, y_prob = predict(model, data['test_ds'])
    metrics = compute_metrics(y_true, y_pred, y_prob)
    print('test metrics:')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')

    # LESSON 11: persist the raw predictions BEFORE anything is derived from
    # them, so a diagnostic invented later can be applied without retraining.
    pred_path = save_predictions(run_name, y_true, y_pred, y_prob,
                                 meta={'arch_variant': arch_variant,
                                       'split_variant': split_variant,
                                       'config_note': config_note})
    print('\npredictions saved to:', pred_path)

    # detect_collapse now also flags partial collapse (LESSON 11): a model that
    # never predicts some classes at all, even with a nonzero kappa.
    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)
    print()
    print_collapse_report(collapse, run_name)

    plot_history(history, run_name)
    plot_confusion_matrix(y_true, y_pred, run_name)
    print('\nper-class report:')
    print(per_class_report(y_true, y_pred).round(4))

    breakdown = tumor_vs_subtype_breakdown(y_true, y_pred)
    print('\ntumour-vs-subtype breakdown:')
    for k, v in breakdown.items():
        print(f'  {k}: {v}')

    row = result_row_from_metrics(
        model_name=run_name, metrics=metrics, collapse=collapse,
        arch_variant=arch_variant, split_variant=split_variant,
        params=count_params(model)['total_params'],
        epochs_trained=final_epoch_summary(history)['epochs_trained'],
        notes=f"binary_tumor_acc={breakdown['binary_tumor_vs_healthy_accuracy']:.4f}; "
              f"subtype_acc={breakdown['subtype_accuracy_all_tumors']:.4f}",
        config_note=config_note if kind == 'experiment' else None)
    path = record_experiment(row) if kind == 'experiment' else None
    print('\nlogged to:', path)
    return {'metrics': metrics, 'collapse': collapse, 'breakdown': breakdown,
            'y_true': y_true, 'y_pred': y_pred}

## 3. Load both splits

**Looks right**: `faithful` ~613/72/315, `clean` smaller and imbalanced with a printed class weight
dict.

In [ ]:
data_faithful = prepare_run('faithful')

In [ ]:
data_clean = prepare_run('clean')

## Run A - `gap` head, `faithful` split

**Looks right**: validation accuracy climbing well above 0.25 within ~10-15 epochs. Because the GAP
head has only ~8.5K parameters after the backbone, this variant tends to train more slowly than the
`flatten` one - that is expected, not a bug.

In [ ]:
run_a, model_a, hist_a = train_run('gap', 'faithful', data_faithful)

In [ ]:
res_a = evaluate_run(run_a, model_a, hist_a, data_faithful, 'gap', 'faithful',
                     config_note='GAP head, faithful split, single run, Adam(1e-4, clipnorm=1.0)')

## Run B - `flatten` head, `faithful` split

This is the variant that matches the paper's stated ~0.5M parameters, so it is the closest
single-run comparison to the published number.

**Looks right**: same sanity criteria as run A; the collapse check must PASS before you take the
accuracy seriously.

In [ ]:
run_b, model_b, hist_b = train_run('flatten', 'faithful', data_faithful)

In [ ]:
res_b = evaluate_run(run_b, model_b, hist_b, data_faithful, 'flatten', 'faithful',
                     config_note='Flatten head (~0.5M params, paper-sized), faithful split, single run')

### Run B follow-up — how big is the `flatten` model actually on disk? (LESSON 2)

The paper claims "~0.5M parameters, ~6MB", but 499,476 float32 weights is only ~1.9MB. Adam keeps
two extra buffers per parameter (momentum + variance), so a checkpoint saved with the optimizer in
it should cost roughly 3× the weights — ~5.9MB, which is what "~6MB" would mean if the authors had
used Keras's default `model.save()` rather than `save_weights()`.

The cell below measures both instead of arguing about it. It needs a **trained** model — Adam's slot
variables are created lazily, so an untrained model has no optimizer state to serialise and the two
files would come out the same size for the wrong reason. `optimizer_slot_scalars_found` in the
printout is the guard against that: if it prints 0, the model never took a training step.

This check is independent of the collapse detection — a partially collapsed model has exactly the
same file size as a good one, so it is valid to run it on run B regardless of run B's status.

In [ ]:
size_check = measure_model_file_sizes(model_b, run_name=run_b)

paper_claim_mb = 6.0
tolerance_mb = 1.0
weights_only = size_check['weights_only_MB']
with_optimizer = size_check['full_save_MB']

print()
if size_check['optimizer_slot_scalars_found'] == 0:
    print('INCONCLUSIVE: no Adam slot variables were found, so the "full" save carries no '
          'optimizer state. Train the model before running this check.')
elif abs(with_optimizer - paper_claim_mb) <= tolerance_mb and weights_only < paper_claim_mb - tolerance_mb:
    print(f'CONFIRMED: weights alone are {weights_only} MB, far short of the paper\'s ~{paper_claim_mb} MB, '
          f'while the default save() including optimizer state is {with_optimizer} MB - within '
          f'{tolerance_mb} MB of the claim. The paper\'s figure is consistent with a save() that '
          f'includes the optimizer.')
elif abs(weights_only - paper_claim_mb) <= tolerance_mb:
    print(f'REFUTED: the weights-only file is already {weights_only} MB, itself close to the '
          f'paper\'s ~{paper_claim_mb} MB, so optimizer state is not needed to explain the claim.')
else:
    print(f'REFUTED: neither file matches the paper\'s ~{paper_claim_mb} MB '
          f'(weights only {weights_only} MB, with optimizer {with_optimizer} MB). '
          f'The stated size must come from something else - a different precision, a different '
          f'serialisation format, or a different architecture than the one reconstructed here.')
print('\nCopy the two measured numbers into README lesson 2 - do not paraphrase them.')

## Run C - `gap` head, `clean` split

**Robustness experiment, not a replication.** The clean split removes the duplicate-driven
train/test leakage, so accuracy here is expected to be *lower* than on `faithful` - that drop is the
result, not a failure.

**Looks right**: inverse-frequency class weights printed above are actually in use, and the model
does not simply ignore the small `normal` class (check the confusion matrix).

In [ ]:
run_c, model_c, hist_c = train_run('gap', 'clean', data_clean)

In [ ]:
res_c = evaluate_run(run_c, model_c, hist_c, data_clean, 'gap', 'clean',
                     config_note='GAP head, CLEAN split (deduplicated, leakage-free) - robustness '
                                 'experiment, NOT a paper replication; class weights on')

## Run D - `flatten` head, `clean` split

Same caveat as run C: robustness experiment, not a replication.

In [ ]:
run_d, model_d, hist_d = train_run('flatten', 'clean', data_clean)

In [ ]:
res_d = evaluate_run(run_d, model_d, hist_d, data_clean, 'flatten', 'clean',
                     config_note='Flatten head, CLEAN split (deduplicated, leakage-free) - robustness '
                                 'experiment, NOT a paper replication; class weights on')

## 4. Side-by-side summary of the four runs

**Looks right**: four rows, all with `status = ok`. Any row tagged `INVALID_collapsed` must be
re-run (lower the learning rate further, or check the input pipeline in notebook 01) - do not quote
its accuracy anywhere.

In [ ]:
summary = pd.DataFrame([
    {'run': run_a, 'arch': 'gap',     'split': 'faithful', **res_a['metrics'], 'status': res_a['collapse']['status']},
    {'run': run_b, 'arch': 'flatten', 'split': 'faithful', **res_b['metrics'], 'status': res_b['collapse']['status']},
    {'run': run_c, 'arch': 'gap',     'split': 'clean',    **res_c['metrics'], 'status': res_c['collapse']['status']},
    {'run': run_d, 'arch': 'flatten', 'split': 'clean',    **res_d['metrics'], 'status': res_d['collapse']['status']},
])
print(summary[['run', 'arch', 'split', 'accuracy', 'f1_macro', 'cohen_kappa', 'mcc', 'status']].round(4).to_string(index=False))

collapsed = summary[summary['status'] != VALID_TAG]
if len(collapsed):
    print('\n!!! collapsed runs (do NOT report these numbers):')
    print(collapsed[['run', 'status']].to_string(index=False))
else:
    print('\nAll four runs passed the collapse check.')

In [ ]:
from src.evaluate_utils import load_results

log = load_results('experiment')
print(f'experiments_log.csv now has {len(log)} rows')
print(log[['model', 'arch_variant', 'split_variant', 'accuracy', 'status']].tail(4).to_string(index=False))
print('\nnext: 03_ablation_dropout.ipynb, then 04_cross_validation.ipynb '
      '(which produces the canonical MiniConvNet number).')